# Notebook C — Evaluation: Zero-shot vs. SemSeg (environment classification)

Compares the two environment classifiers on a hand-labeled **multi-label** test set:

- **Accuracy:** macro **F1** + accuracy (subset / per-label) per method, and per-class F1.
- **Performance:** mean **runtime per frame** (from each notebook's run).

The better overall method is chosen for environment classification; per-class F1 also shows whether
a **combination** (pick the better method per class) would help.


## 1. Setup & paths

In [ ]:
import sys, json
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd()))
import segmentation_common as sc

ENV_CLASSES = sc.CATEGORIES["environment"]
EVAL_DIR = Path("../dataset/eval")
TEST_DIR = Path("../dataset/test_images")

# Ground truth = hand-labeled multi-hot CSV from scripts/label_tool.py
# (key: filename == path relative to dataset/test_images/).
LABELS_CSV = TEST_DIR / "labels.csv"
# Prediction CSVs now hold CONTINUOUS SCORES (CLIP softmax prob / SegFormer
# pixel-area fraction). This cell binarizes them with per-method thresholds.
PRED = {"zeroshot": EVAL_DIR / "env_pred_zeroshot.csv",
        "semseg":   EVAL_DIR / "env_pred_semseg.csv"}
RUNTIME = {"zeroshot": EVAL_DIR / "runtime_zeroshot.json",
           "semseg":   EVAL_DIR / "runtime_semseg.json"}

# Operating-point thresholds, aligned to the annotation rules:
#   zeroshot: CLIP softmax present-prob >= 0.5
#   semseg:   pixel-area fraction; water 0.02 == the ">2% water" rule; the
#             others are dev-tuned proxies (segmentation gives area, not a tree
#             count / sky test) - a sweep below shows their achievable ceiling.
THRESHOLDS = {   # 3-class, val-tuned (tune_thresholds.py)
    "zeroshot": {"vegetation": 0.035, "water": 0.39, "city": 0.73},
    "semseg":   {"vegetation": 0.25, "water": 0.01, "city": 0.01},
}
print("environment classes:", ENV_CLASSES)
print("thresholds:", THRESHOLDS)

## 2. Prepare the test-label template

The test set holds ~150 frames **per class** (multi-label: a frame may belong to several classes).
Labeling happens separately — run this once to scaffold an empty `env_labels.csv` from the images
in `dataset/eval/images/`, then fill each class column with 0/1.

In [ ]:
# Labels now come from scripts/label_tool.py (multi-hot + unsure + reject),
# not a template generated here. This cell just reports labeling progress.
def labeling_status():
    if not LABELS_CSV.exists():
        print("No labels yet - run: python scripts/label_tool.py")
        return
    df = pd.read_csv(LABELS_CSV)
    n_all = len(list(TEST_DIR.rglob("*.jpg"))) + len(list(TEST_DIR.rglob("*.png")))
    rej = int(df.get("reject", pd.Series(0, index=df.index)).fillna(0).sum())
    uns = int(df.get("unsure", pd.Series(0, index=df.index)).fillna(0).sum())
    kept = len(df) - rej
    print(f"labeled {len(df)}/{n_all}  |  rejected(non-POV) {rej}  |  unsure {uns}  "
          f"|  usable for scoring {kept - uns}")
    src = df["filename"].str.split("/").str[0].replace({"own_frames": "own"})
    print("by source:", src.value_counts().to_dict())


labeling_status()

## 3. Load ground truth + predictions

Aligns each method's predictions to the labeled rows and returns 0/1 arrays over `ENV_CLASSES`.

In [ ]:
import csv as _csv, re as _re
_OWN_SPLIT = {r["ride_id"]: r["split"] for r in
              _csv.DictReader(open(EVAL_DIR / "own_split.csv"))} \
             if (EVAL_DIR / "own_split.csv").exists() else {}


def source_of(fn: str) -> str:
    top = fn.split("/")[0]
    if top != "own_frames":
        return top                                    # ade20k | mapillary
    m = _re.search(r"(DJI_\d+)", fn)                   # own frames split by ride
    return "own-" + _OWN_SPLIT.get(m.group(1), "test") if m else "own-test"


def load_eval():
    """Return {method: {y_true, score, y_pred, source, files}} over images that
    are labeled, not reject, not unsure, and predicted. score = continuous;
    y_pred = score >= per-class threshold."""
    gt = pd.read_csv(LABELS_CSV)
    for col in ("reject", "unsure"):
        if col not in gt.columns:
            gt[col] = 0
    keep = (gt["reject"].fillna(0).astype(int) == 0) & \
           (gt["unsure"].fillna(0).astype(int) == 0)
    gt = gt[keep].set_index("filename")
    gt_bin = gt.reindex(columns=ENV_CLASSES).fillna(0).astype(int)

    methods = {}
    for name, p in PRED.items():
        if not p.exists():
            continue
        sc_df = (pd.read_csv(p).set_index("filename")
                 .reindex(columns=ENV_CLASSES).fillna(0).astype(float))
        common = gt_bin.index.intersection(sc_df.index)
        if len(common) == 0:
            continue
        score = sc_df.loc[common].values
        thr = np.array([THRESHOLDS[name][c] for c in ENV_CLASSES])
        methods[name] = {
            "y_true": gt_bin.loc[common].values,
            "score": score,
            "y_pred": (score >= thr).astype(int),
            "source": np.array([source_of(f) for f in common]),
            "files": list(common),
        }
    return methods

## 4. Accuracy metrics for both approaches  ←  the comparison cell

Computes **macro-F1**, subset accuracy, per-label accuracy and **per-class F1** for each method.
Falls back to a small synthetic demo if the labels/predictions don't exist yet.

In [ ]:
def evaluate():
    methods = load_eval()
    if not methods:
        print("No labeled+predicted images yet.")
        return None, None, None
    overall, perclass, per_source = [], {}, []
    for name, d in methods.items():
        m = sc.multilabel_metrics(d["y_true"], d["y_pred"])
        overall.append({"method": name, "n": len(d["y_true"]),
                        "macro_F1": round(m["macro_f1"], 3),
                        "label_accuracy": round(m["label_accuracy"], 3),
                        "subset_accuracy": round(m["subset_accuracy"], 3)})
        perclass[name] = m["per_class_f1"]
        for s in ("own-test", "own-val", "ade20k", "mapillary"):
            mask = (d["source"] == s)
            if mask.sum() == 0:
                continue
            ms = sc.multilabel_metrics(d["y_true"][mask], d["y_pred"][mask])
            row = {"method": name, "source": s, "n": int(mask.sum()),
                   "macro_F1": round(ms["macro_f1"], 3),
                   "label_acc": round(ms["label_accuracy"], 3)}
            for j, c in enumerate(ENV_CLASSES):
                row[f"F1_{c}"] = round(float(ms["per_class_f1"][j]), 3)
            per_source.append(row)
    return (pd.DataFrame(overall),
            pd.DataFrame(perclass, index=ENV_CLASSES).round(3),
            pd.DataFrame(per_source))


overall, per_class_f1, per_source = evaluate()
if overall is not None:
    print("== Overall (all sources; reject/unsure excluded; operating thresholds) ==")
    display(overall)
    print("== Per-class F1 (all sources) ==")
    display(per_class_f1)
    print("== Per-source (own = target cyclist-POV domain) ==")
    display(per_source.sort_values(["source", "method"]))

## 5. Performance: runtime per frame

In [ ]:
rt_rows = []
for name, path in RUNTIME.items():
    if path.exists():
        d = json.load(open(path))
        rt_rows.append({"method": name, "ms_per_frame": round(d.get("ms_per_frame", d.get("ms_per_frame_both", float("nan"))), 1),
                        "n": d.get("n"), "value": d.get("value")})
runtime_table = pd.DataFrame(rt_rows) if rt_rows else pd.DataFrame(
    columns=["method", "ms_per_frame", "fps", "n"])
if rt_rows:
    display(runtime_table)
else:
    print("Run Notebooks A and B first to produce runtime_*.json")

In [ ]:
# Per-class threshold sweep on the labeled data = ACHIEVABLE CEILING per
# method/class (optimistic: tuned on the same labels). Shows how much of any gap
# is decision-threshold vs. the model itself.
def sweep(source=None):
    methods = load_eval()
    grid = np.r_[np.linspace(0.005, 0.1, 20), np.linspace(0.12, 0.9, 20)]
    out = {}
    for name, d in methods.items():
        yt, scv, src = d["y_true"], d["score"], d["source"]
        m = np.ones(len(yt), bool) if source is None else (src == source)
        yt, scv = yt[m], scv[m]
        best = []
        for j in range(len(ENV_CLASSES)):
            f1s = []
            for t in grid:
                yp = (scv[:, j] >= t).astype(int)
                tp = int((yp & yt[:, j]).sum()); fp = int((yp & (1-yt[:, j])).sum())
                fn = int(((1-yp) & yt[:, j]).sum())
                f1s.append((2*tp/max(2*tp+fp+fn,1), t))
            best.append(max(f1s))
        out[name] = {ENV_CLASSES[j]: round(best[j][0],3) for j in range(len(ENV_CLASSES))}
        out[name]["macro"] = round(float(np.mean([b[0] for b in best])),3)
        out[name]["thr*"] = {ENV_CLASSES[j]: round(best[j][1],3) for j in range(len(ENV_CLASSES))}
    return pd.DataFrame({k:{kk:vv for kk,vv in v.items() if kk!="thr*"} for k,v in out.items()}).T, out

print("== Ceiling on OWN-TEST frames (best per-class threshold, optimistic) ==")
tbl, raw = sweep(source="own-test")
display(tbl)
for m in raw: print(f"  {m} best thresholds:", raw[m]["thr*"])

## 6. Verdict & possible combination

- **Best overall method** = higher `macro_F1` at acceptable `ms_per_frame`.
- **Combination check:** if neither method has the best F1 on *every* class, a hybrid that takes
  each class from its stronger method could beat both — quantified below.

In [ ]:
if per_class_f1 is not None and per_class_f1.shape[1] >= 2:
    best_per_class = per_class_f1.idxmax(axis=1)
    hybrid_f1 = per_class_f1.max(axis=1).mean()
    print("Best method per class:")
    print(best_per_class.to_string())
    print(f"\nMacro-F1 if combined (best per class): {hybrid_f1:.3f}")
    print("Per-method macro-F1:",
          {m: round(per_class_f1[m].mean(), 3) for m in per_class_f1.columns})
    print("\n-> A combination helps only if the per-class winner is split across methods.")
else:
    print("Need both methods' predictions (and labels) to assess a combination.")